In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

# pep_portfolio
pep_portfolio_schema = StructType([
    StructField("pe_firm", StringType(), True),
    StructField("company", StringType(), True),
    StructField("shares", IntegerType(), True)
])

pep_portfolio_data = [
    ("Alpha", "A", 1000),
    ("Alpha", "B", 2000),
    ("Beta", "A", 1500),
    ("Beta", "C", 2500),
    ("Gamma", "B", 1200),
    ("Gamma", "C", 1300)
]

pep_portfolio_df = spark.createDataFrame(
    pep_portfolio_data,
    schema=pep_portfolio_schema
)

# pep_prices
pep_prices_schema = StructType([
    StructField("date", DateType(), True),
    StructField("company", StringType(), True),
    StructField("closing_price", IntegerType(), True)
])

pep_prices_data = [
    ("2023-01-01", "A", 50),
    ("2023-01-01", "B", 20),
    ("2023-01-01", "C", 30),
    ("2023-01-02", "A", 52),
    ("2023-01-02", "B", 21),
    ("2023-01-02", "C", 31)
]

pep_prices_df = spark.createDataFrame(
    pep_prices_data,
    schema=["date", "company", "closing_price"]
).withColumn("date", to_date(col("date")))

In [0]:
result_df = (
    pep_portfolio_df.join(pep_prices_df, on="company", how="inner")
    .groupBy("pe_firm", "date")
    .agg(sum(col("shares") * col("closing_price")).alias("portfolio_value"))
    .select(
        col("pe_firm").alias("PE_firm"),
        date_format(col("date"), "yyyy-MM-dd").alias("date"),
        col("portfolio_value"),
    )
)
display(result_df)